In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

In [22]:
auditors = pd.read_csv('auditors.csv')
restatements = pd.read_csv('restatements.txt', sep = ',')

In [120]:
big4_key = ['Deloitte & Touche LLP', 'KPMG LLP', 'PricewaterhouseCoopers LLP', 'Ernst & Young LLP']

# changing auditors and restatements to appropriate column types, date range
auditors['FISCAL_YEAR'] = pd.to_datetime(auditors['FISCAL_YEAR'], format = '%Y')
auditors['year'] = auditors['FISCAL_YEAR'].dt.year
auditors = auditors[(auditors['year'] >= 2008) & (auditors['year'] <= 2009)]
auditors['big4_bool'] = auditors['AUDITOR_NAME'].isin(big4_key)
auditors.rename(columns = {'COMPANY_FKEY': 'CompanyId'}, inplace = True)

restatements['FilingDate'] = pd.to_datetime(restatements['FilingDate'], format = '%m/%d/%Y')
restatements['year'] = restatements['FilingDate'].dt.year
restatements = restatements[(restatements['year'] >= 2008) & (restatements['year'] <= 2009)]

# columns for total number of restatements by company, filters for companies with any restatement, multiple restatements
count = restatements.groupby(['CompanyId', 'year']).size().reset_index(name = 'n_restatements')
count['any_restatement'] = (count['n_restatements'] > 0).astype(int)
count['multi_restatements'] = (count['n_restatements'] > 1).astype(int)
df = auditors.merge(count, on = ['CompanyId', 'year'], how = 'left')

df['n_restatements'] = df['n_restatements'].fillna(0).astype(int)
df['any_restatement'] = df['any_restatement'].fillna(0).astype(int)
df['multi_restatements'] = df['multi_restatements'].fillna(0).astype(int)

# Descriptives

#print(df.sort_values(by = 'n_restatements', ascending = False))
#print(df.groupby('big4_bool')['n_restatements'].sum())
#print(df.groupby('big4_bool')['any_restatement'].sum())
print(df.groupby('big4_bool')['any_restatement'].agg(['sum', 'count', 'mean']))
print(df.groupby('big4_bool')['multi_restatements'].agg(['sum', 'count', 'mean']))

# Hyp. tests
# Is restatement status independent of Big4 status? i.e. H_0: big 4 proportion = not big 4 proportion

ct = pd.crosstab(df['big4_bool'], df['any_restatement'])
chi2, p, dof, expected = chi2_contingency(ct)
print('p:', p) 

# p: .010244598026038046 => probably H_a, specifically big 4 proportion < not big 4 proportion
# equivalent to two-sided z-test

# don't have sector information for nonrestating firms and there's a weird firm I don't want to deal with, see below

'''
sector_check = (
    restatements
      .groupby(['CompanyId', 'year'])['SectorName']
      .nunique()
      .reset_index(name='n_sectors')
)
sector_check[sector_check['n_sectors'] > 1]

restatements[restatements['CompanyId'] == 1134203]
'''

           sum  count      mean
big4_bool                      
False      204  11710  0.017421
True       183  13676  0.013381
           sum  count      mean
big4_bool                      
False       12  11710  0.001025
True         5  13676  0.000366
p: 0.010244598026038046


"\nsector_check = (\n    restatements\n      .groupby(['CompanyId', 'year'])['SectorName']\n      .nunique()\n      .reset_index(name='n_sectors')\n)\nsector_check[sector_check['n_sectors'] > 1]\n\nrestatements[restatements['CompanyId'] == 1134203]\n"

In [23]:
restatements = pd.read_csv('restatements.txt', sep = ',')
restatements['FilingDate'] = pd.to_datetime(restatements['FilingDate']) # convert to dt
restatements_sorted = restatements.sort_values('FilingDate') # sorting so that I can select entries using 'first' later

# run on .groupby() with .apply() to find the date stats for each group in groupedby()
def date_stats(group):
    g = group.sort_values('FilingDate')
    dates = g['FilingDate'].dropna()

    # if the number of dates in a group is one we can't deltas but the MaxDays calculation will still return 0 instead of nothing while MinDays and MedianDays will
    if len(dates) < 2:
        return pd.Series({
            'MinDays' : pd.NA,
            'MedianDays' : pd.NA,
            'MaxDays' : pd.NA
        })
    
    diffs = dates.diff().dt.days.dropna()   
    min_diff = diffs.min()
    median_diff = diffs.median()
    max_diff = (g['FilingDate'].max() - g['FilingDate'].min()).days

    return pd.Series({
        'MinDays' : min_diff,
        'MedianDays' : median_diff,
        'MaxDays' : max_diff
    })

# df with the above function for a table with the columns to merge
date_stats = (
    restatements
      .groupby(['SectorId', 'RestatementType1'])
      .apply(date_stats)
)

restatements_new = restatements_sorted.groupby(['SectorId', 'RestatementType1']).agg(
        SectorName = ('SectorName', 'first'),
        CompanyCount = ('CompanyId', 'size'),
        CompanyID = ('CompanyId', 'first'),
        CompanyName = ('CompanyName', 'first'),
        FirstRestateDate = ('FilingDate', 'min')
).reset_index()

# moving SectorName forward
restatements_new = restatements_new.merge(date_stats, on=['SectorId', 'RestatementType1'], how='left')
restatements_new = restatements_new[
    [
        'SectorId',
        'SectorName',
        'RestatementType1',
        'CompanyCount',
        'CompanyID',
        'CompanyName',
        'FirstRestateDate',
        'MinDays',
        'MedianDays',
        'MaxDays',
    ]
]

print(restatements_new)

latex = restatements_new.to_latex(index=False, float_format="%.2f")
with open('restatement_summary.tex', 'w', encoding='utf-8') as f:
    f.write(latex)

    SectorId             SectorName RestatementType1  CompanyCount  CompanyID  \
0          1        Basic Materials               ER             8    1096841   
1          1        Basic Materials              OTH            27    1288750   
2          1        Basic Materials               RR             2    1014669   
3          2          Capital Goods               ER             6     357108   
4          2          Capital Goods              OTH            20      80035   
5          2          Capital Goods               RR             2      65201   
6          3      Consumer Cyclical               ER             4    1160858   
7          3      Consumer Cyclical              OTH            21     923168   
8          3      Consumer Cyclical               RR             1    1046692   
9          4  Consumer Non-Cyclical               ER             2     910406   
10         4  Consumer Non-Cyclical              OTH            11     707674   
11         4  Consumer Non-C

C:\Users\Limes\AppData\Local\Temp\ipykernel_20068\1606177809.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(date_stats)
